In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Qwen3ForCausalLM

from llminfer.models.qwen3 import Qwen3ForCausalLM as OurQwen3ForCausalLM

/Users/ericchen/Eric/llm-infer/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
default_qwen3 = Qwen3ForCausalLM.from_pretrained("Qwen/Qwen3-0.6B", dtype=torch.float32, attn_implementation="sdpa")
our_qwen3 = OurQwen3ForCausalLM.from_pretrained("Qwen/Qwen3-0.6B")

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 3114.55it/s]


In [5]:
device = our_qwen3.device
messages = [{"role": "user", "content": "Hi, my name is"}]
text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
)
inputs = tokenizer([text], return_tensors="pt").to(device)
input_ids, attention_mask = inputs["input_ids"], inputs["attention_mask"]

In [8]:
default_qwen3.eval()
with torch.no_grad():
    outputs = default_qwen3(input_ids, attention_mask=attention_mask, output_hidden_states=True)

# outputs.hidden_states[-1]
outputs.logits

tensor([[[ 3.6899,  3.8583,  3.7428,  ...,  1.7098,  1.7098,  1.7098],
         [ 7.3151,  8.3569,  6.4606,  ...,  0.4270,  0.4270,  0.4270],
         [ 4.6694,  9.6475, 11.5166,  ...,  3.8347,  3.8347,  3.8347],
         ...,
         [ 7.6073, 15.7317,  8.9375,  ...,  1.1540,  1.1540,  1.1540],
         [-3.5693, -3.1510, -7.4014,  ..., -1.2693, -1.2693, -1.2693],
         [11.1866, 13.3352,  4.5635,  ...,  3.1612,  3.1612,  3.1612]]])

In [9]:
our_qwen3.eval()
with torch.no_grad():
    outputs = our_qwen3(input_ids, attention_mask=attention_mask)

outputs.logits

tensor([[[ 3.6899,  3.8583,  3.7428,  ...,  1.7098,  1.7098,  1.7098],
         [ 7.3152,  8.3569,  6.4607,  ...,  0.4270,  0.4270,  0.4270],
         [ 4.6695,  9.6475, 11.5166,  ...,  3.8347,  3.8347,  3.8347],
         ...,
         [ 7.6073, 15.7317,  8.9375,  ...,  1.1540,  1.1540,  1.1540],
         [-3.5693, -3.1510, -7.4014,  ..., -1.2693, -1.2693, -1.2693],
         [11.1866, 13.3353,  4.5635,  ...,  3.1612,  3.1612,  3.1612]]])

In [9]:
logits = outputs.logits[0, -1, :]
probs = logits.softmax(dim=-1)
next_token_id = torch.multinomial(probs, num_samples=1)
print(next_token_id)

next_token = tokenizer.decode(next_token_id[0])
print(next_token)

tensor([9707])
Hello
